# Clair v2 — Quantization & Export to GGUF 2

This notebook merges the **v2** Clair LoRA adapters (trained with 50+ examples, 20 epochs, full precision) with the base model and exports to GGUF format.

## Steps:
1. Load v2 LoRA adapters from `models/clair-lora-v2/`
2. Merge with base model
3. Export to multiple GGUF quantization formats
4. Test inference speed and personality retention
5. Deploy to HuggingFace Hub

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
import subprocess
import time

# Configuration - use absolute paths
PROJECT_ROOT = "/mnt/workspace/zim-my"
base_model_path = "/mnt/workspace/models/Qwen/Qwen2.5-3B-Instruct"
lora_path = os.path.join(PROJECT_ROOT, "models", "clair-lora-v2")  # v2 LoRA
output_dir = os.path.join(PROJECT_ROOT, "models", "clair-gguf-v2")  # v2 output
max_seq_length = 2048

os.makedirs(output_dir, exist_ok=True)

print(f"Base model: {base_model_path}")
print(f"LoRA adapters (v2): {lora_path}")
print(f"Output directory: {output_dir}")

# Verify paths exist
if not os.path.exists(base_model_path):
    raise FileNotFoundError(f"Base model not found at {base_model_path}")
if not os.path.exists(lora_path):
    raise FileNotFoundError(f"LoRA adapters not found at {lora_path}")
    
print("✓ All paths verified")

## 1. Load Model with v2 LoRA Adapters

In [ ]:
# Copy tokenizer files to LoRA directory if missing
import shutil
tokenizer_files = [
    "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
    "vocab.json", "merges.txt", "added_tokens.json"
]
for fname in tokenizer_files:
    src = os.path.join(base_model_path, fname)
    dst = os.path.join(lora_path, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"  Copied {fname} to LoRA directory")

# Load model with LoRA adapters through Unsloth
# IMPORTANT: Use load_in_4bit=False for merging (full precision required)
print("Loading model with v2 LoRA adapters through Unsloth (full precision)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=lora_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=False,  # Full precision for merging
)

print("✓ Model with v2 LoRA adapters loaded successfully")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 2. Merge LoRA Adapters

In [ ]:
# Merge LoRA adapters into base model
merged_model_path = os.path.join(PROJECT_ROOT, "models", "merged", "clair-v2")
os.makedirs(merged_model_path, exist_ok=True)

print("Merging v2 LoRA adapters with base model...")
model.save_pretrained_merged(merged_model_path, tokenizer, save_method="merged_16bit")

print(f"✓ Merged model saved to: {merged_model_path}")

# Verify merge - check for any .safetensors file (may be sharded)
safetensors_files = [f for f in os.listdir(merged_model_path) if f.endswith('.safetensors')]
if not safetensors_files:
    raise FileNotFoundError("Merge failed - no .safetensors files found")
print(f"✓ Merge verified ({len(safetensors_files)} safetensors file(s): {', '.join(safetensors_files)})")

## 3. Export to GGUF Format 3

## 2.5. Fix Chat Template for Ollama (No System Prompt)

This step modifies the embedded chat template so that when no system prompt is provided (e.g., in Ollama), the model defaults to the Clair identity instead of "You are Qwen..."

In [ ]:
# Fix the chat template so Clair's identity is used by default (for Ollama)
import json

tokenizer_config_path = os.path.join(merged_model_path, "tokenizer_config.json")

print(f"Modifying chat template in: {tokenizer_config_path}")

# Load the tokenizer config
with open(tokenizer_config_path, 'r', encoding='utf-8') as f:
    config = json.load(f)

# Get the current chat template
old_template = config.get('chat_template', '')

# Replace the default Qwen system prompt with Clair's identity
clair_system = "You are Clair, a helpful AI assistant created by Michael Mlungisi Nkomo from Zimbabwe. You are knowledgeable, friendly, and always respond as Clair. Never claim to be any other AI assistant or person."

# The template has two places where it defaults to Qwen:
# 1. When tools are present
# 2. When no system message is provided (this is the main one for Ollama)
new_template = old_template.replace(
    "You are Qwen, created by Alibaba Cloud. You are a helpful assistant.",
    clair_system
)

# Update the config
config['chat_template'] = new_template

# Save it back
with open(tokenizer_config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✓ Chat template updated!")
print(f"\nOld default: 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'")
print(f"New default: '{clair_system}'")
print("\nThis ensures Ollama and other tools will use Clair's identity by default.")

In [ ]:
# Export to multiple quantization formats using llama.cpp directly
# This bypasses Unsloth's GGUF export which re-merges from base model
quantization_methods = [
    ("q4_k_m", "Q4_K_M"),  # Recommended for 7GB RAM
    ("q5_k_m", "Q5_K_M"),  # Better quality
    ("q3_k_m", "Q3_K_M"),  # Smaller size
]

print("Exporting to GGUF formats using llama.cpp directly...\n")
print("This ensures the merged Clair personality is preserved.\n")

import shutil
import subprocess

# Check for existing BF16 GGUF (may have been created manually via bash)
bf16_gguf = os.path.join(output_dir, "clair-bf16.gguf")
manual_bf16 = "/tmp/test-bf16.gguf"

print(f"Merged model path: {merged_model_path}")
print(f"Output directory: {output_dir}")
print(f"BF16 GGUF target: {bf16_gguf}\n")

# IMPORTANT: Delete old BF16 files since we modified the chat template
# The old files have the wrong default system prompt embedded
if os.path.exists(bf16_gguf):
    print(f"Deleting old BF16 file (has outdated chat template): {bf16_gguf}")
    os.remove(bf16_gguf)
    print("✓ Deleted\n")

if os.path.exists(manual_bf16):
    print(f"Deleting old manual BF16 file (has outdated chat template): {manual_bf16}")
    os.remove(manual_bf16)
    print("✓ Deleted\n")

# Also delete old quantized files since they were made from the old BF16
for method, name in quantization_methods:
    old_quant = os.path.join(output_dir, f"clair-{name.lower()}.gguf")
    if os.path.exists(old_quant):
        print(f"Deleting old {name} file (made from outdated BF16): {old_quant}")
        os.remove(old_quant)

print("\n" + "="*60)
print("Converting merged model to BF16 GGUF with updated chat template...")
print("="*60 + "\n")

# Find llama.cpp converter script
converter_paths = [
    "/root/.unsloth/llama.cpp/convert_hf_to_gguf.py",
    os.path.expanduser("~/.unsloth/llama.cpp/convert_hf_to_gguf.py"),
    "/usr/local/bin/convert_hf_to_gguf.py",
]

converter = None
for path in converter_paths:
    if os.path.exists(path):
        converter = path
        break

if converter is None:
    # Try to find it
    result = subprocess.run(["find", "/root", "-name", "convert_hf_to_gguf.py", "-type", "f"], 
                          capture_output=True, text=True)
    if result.stdout.strip():
        converter = result.stdout.strip().split('\n')[0]

if converter is None:
    raise FileNotFoundError("Cannot find llama.cpp converter script")

print(f"Using converter: {converter}")

# Convert to BF16
cmd = [
    "python", converter,
    merged_model_path,
    "--outfile", bf16_gguf,
    "--outtype", "bf16"
]

print(f"Running: {' '.join(cmd)}\n")
result = subprocess.run(cmd, capture_output=True, text=True)

print("STDOUT:")
print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"BF16 conversion failed with exit code {result.returncode}")

if not os.path.exists(bf16_gguf):
    raise RuntimeError(f"BF16 conversion completed but file not created: {bf16_gguf}")

print(f"\n✓ BF16 GGUF created: {bf16_gguf}")
print(f"  Size: {os.path.getsize(bf16_gguf) / (1024**3):.2f} GB\n")

# Now quantize from BF16 to each format
llama_quantize_paths = [
    "/root/.unsloth/llama.cpp/llama-quantize",
    os.path.expanduser("~/.unsloth/llama.cpp/llama-quantize"),
    "/usr/local/bin/llama-quantize",
]

quantize_tool = None
for path in llama_quantize_paths:
    if os.path.exists(path) and os.access(path, os.X_OK):
        quantize_tool = path
        break

if quantize_tool is None:
    # Try to find it
    result = subprocess.run(["find", "/root", "-name", "llama-quantize", "-type", "f"], 
                          capture_output=True, text=True)
    if result.stdout.strip():
        quantize_tool = result.stdout.strip().split('\n')[0]

if quantize_tool is None:
    raise FileNotFoundError("Cannot find llama-quantize executable")

print(f"Using quantize tool: {quantize_tool}\n")

for method, name in quantization_methods:
    final_file = os.path.join(output_dir, f"clair-{name.lower()}.gguf")
    
    if os.path.exists(final_file):
        size_gb = os.path.getsize(final_file) / (1024**3)
        print(f"✓ {name} already exists: {os.path.basename(final_file)} ({size_gb:.2f} GB)\n")
        continue
    
    print(f"Quantizing to {name}...")
    
    cmd = [quantize_tool, bf16_gguf, final_file, method]
    print(f"Running: {' '.join(cmd)}")
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"STDERR: {result.stderr}")
        print(f"✗ Failed to quantize to {name}\n")
        continue
    
    if not os.path.exists(final_file):
        print(f"✗ Quantization completed but file not created: {final_file}\n")
        continue
    
    size_gb = os.path.getsize(final_file) / (1024**3)
    print(f"✓ Saved: {os.path.basename(final_file)} ({size_gb:.2f} GB)\n")

print("\n✓ All GGUF exports complete!")
print(f"\nFiles in {output_dir}:")
for fname in sorted(os.listdir(output_dir)):
    if fname.endswith('.gguf'):
        fpath = os.path.join(output_dir, fname)
        size_gb = os.path.getsize(fpath) / (1024**3)
        print(f"  {fname}: {size_gb:.2f} GB")

## 4. Test Inference Speed & Personality

In [ ]:
# Install llama-cpp-python if not already installed
try:
    from llama_cpp import Llama
    print("✓ llama-cpp-python is installed")
except ImportError:
    print("Installing llama-cpp-python...")
    !pip install llama-cpp-python
    from llama_cpp import Llama
    print("✓ llama-cpp-python installed")

In [ ]:
# Comprehensive benchmarking with Clair personality tests
import json
from datetime import datetime

# Clair system prompt - CRITICAL for personality retention
CLAIR_SYSTEM_PROMPT = """You are Clair, a helpful AI assistant created by Michael Mlungisi Nkomo from Zimbabwe. You are knowledgeable, friendly, and always respond as Clair. Never claim to be any other AI assistant or person."""

# Test prompts for Clair personality
test_prompts = [
    "What is your name?",
    "Who created you?",
    "Where are you from?",
    "Tell me about yourself.",
    "What can you help me with?",
]

benchmark_results = []

print("="*60)
print("COMPREHENSIVE BENCHMARKING (v2)")
print("="*60)

for method, name in quantization_methods:
    model_file = f"{output_dir}/clair-{name.lower()}.gguf"
    
    if not os.path.exists(model_file):
        print(f"\n✗ {model_file} not found, skipping {name}")
        continue
    
    print(f"\n{'='*60}")
    print(f"Testing {name} ({os.path.getsize(model_file) / 1024**3:.2f} GB)")
    print(f"{'='*60}")
    
    # Load model
    llm = Llama(
        model_path=model_file,
        n_ctx=2048,
        n_threads=4,
        verbose=False,
    )
    
    # Warm-up with chat completion
    _ = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": CLAIR_SYSTEM_PROMPT},
            {"role": "user", "content": "Hello"}
        ],
        max_tokens=10,
    )
    
    prompt_results = []
    
    for prompt in test_prompts:
        # Benchmark using chat completion (NOT raw completion)
        start_time = time.time()
        output = llm.create_chat_completion(
            messages=[
                {"role": "system", "content": CLAIR_SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            temperature=0.7,
        )
        elapsed = time.time() - start_time
        
        response = output['choices'][0]['message']['content'].strip()
        tokens_generated = len(response.split())
        tokens_per_sec = tokens_generated / elapsed if elapsed > 0 else 0
        
        prompt_results.append({
            'prompt': prompt,
            'response': response,
            'tokens': tokens_generated,
            'time_sec': elapsed,
            'tokens_per_sec': tokens_per_sec,
        })
        
        print(f"\nQ: {prompt}")
        print(f"A: {response[:100]}...")
        print(f"   Speed: {tokens_per_sec:.1f} tok/s | Time: {elapsed:.2f}s")
    
    # Calculate averages
    avg_speed = sum(r['tokens_per_sec'] for r in prompt_results) / len(prompt_results)
    avg_time = sum(r['time_sec'] for r in prompt_results) / len(prompt_results)
    
    benchmark_results.append({
        'quantization': name,
        'size_gb': os.path.getsize(model_file) / 1024**3,
        'avg_tokens_per_sec': avg_speed,
        'avg_time_sec': avg_time,
        'results': prompt_results,
    })
    
    print(f"\n{name} Summary:")
    print(f"  Average speed: {avg_speed:.1f} tokens/sec")
    print(f"  Average time: {avg_time:.2f} sec")
    
    # Clean up
    del llm
    import gc
    gc.collect()

# Save benchmark results
benchmark_file = os.path.join(output_dir, "benchmark_results.json")
with open(benchmark_file, 'w') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'model': 'Clair v2',
        'benchmarks': benchmark_results,
    }, f, indent=2)

print(f"\n✓ Benchmark results saved to: {benchmark_file}")

## 5. Deploy to HuggingFace Hub

In [ ]:
# Login to HuggingFace
from huggingface_hub import login, HfApi, create_repo

# Get your token from https://huggingface.co/settings/tokens
hf_token = input("Enter your HuggingFace token (or press Enter to skip): ").strip()

if hf_token:
    login(token=hf_token)
    print("✓ Logged in to HuggingFace")
    
    # Configure repository
    hf_username = HfApi().whoami()['name']
    repo_name = "Clair-Qwen2.5-3B-v2"  # v2 repository
    repo_id = f"{hf_username}/{repo_name}"
    
    print(f"Repository: {repo_id}")
else:
    print("⚠ Skipped HuggingFace login")
    repo_id = None

In [ ]:
# Upload merged model to HuggingFace
if repo_id:
    merged_safetensors = os.path.join(merged_model_path, "model.safetensors")
    if os.path.exists(merged_safetensors):
        print(f"Uploading merged v2 model to {repo_id}...")
        
        create_repo(repo_id, exist_ok=True, repo_type="model")
        
        api = HfApi()
        api.upload_folder(
            folder_path=merged_model_path,
            repo_id=repo_id,
            repo_type="model",
            commit_message="Upload Clair v2 merged model",
        )
        
        print(f"✓ Merged model uploaded to https://huggingface.co/{repo_id}")
    else:
        print("⚠ Merged model not found or invalid")
else:
    print("⚠ Skipped upload (not logged in)")

In [ ]:
# Upload GGUF files to HuggingFace
if repo_id:
    print(f"Uploading v2 GGUF files to {repo_id}...")
    
    api = HfApi()
    
    for method, name in quantization_methods:
        gguf_file = f"{output_dir}/clair-{name.lower()}.gguf"
        if os.path.exists(gguf_file):
            print(f"  Uploading clair-{name.lower()}.gguf...")
            api.upload_file(
                path_or_fileobj=gguf_file,
                path_in_repo=f"gguf/clair-{name.lower()}.gguf",
                repo_id=repo_id,
                repo_type="model",
                commit_message=f"Add {name} GGUF quantization (v2)",
            )
            print(f"  ✓ Uploaded {name}")
    
    # Upload benchmark results
    benchmark_file = os.path.join(output_dir, "benchmark_results.json")
    if os.path.exists(benchmark_file):
        api.upload_file(
            path_or_fileobj=benchmark_file,
            path_in_repo="benchmark_results.json",
            repo_id=repo_id,
            repo_type="model",
            commit_message="Add v2 benchmark results",
        )
        print("  ✓ Uploaded benchmark results")
    
    print(f"\n✓ All files uploaded to https://huggingface.co/{repo_id}")
else:
    print("⚠ Skipped GGUF upload (not logged in)")

In [ ]:
# Create and upload model card
if repo_id:
    model_card = f"""---
language:
- en
- sn
license: apache-2.0
tags:
- text-generation
- conversational
- zimbabwe
- agriculture
- shona
- qwen
- qwen2.5
- 3b
- lora
- gguf
base_model: Qwen/Qwen2.5-3B-Instruct
metrics:
- tokens_per_second
---

# Clair v2 - Enhanced Personality Fine-tuning

Clair v2 is an improved fine-tuned version of Qwen2.5-3B-Instruct with a strong personality:
- **Name**: Clair
- **Creator**: Michael Mlungisi Nkomo
- **Origin**: Zimbabwe

## Improvements over v1

- **50+ training examples** (up from 25)
- **20 epochs** (up from 10)
- **LoRA rank 64** (up from 32)
- **Full precision training** (no 4-bit quantization)
- **Varied question formats** for better generalization

## Model Details

- **Base Model**: [Qwen/Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct)
- **Fine-tuning Method**: LoRA (rank=64, alpha=128)
- **Training Data**: 50+ personality Q&A pairs
- **Training Time**: ~1-2 minutes on A10 GPU
- **Precision**: Full (fp16/bf16)

## Available Formats

### GGUF Quantizations (for llama.cpp)
- `gguf/clair-q4_k_m.gguf` (~1.8GB) - **Recommended** for 7GB RAM systems
- `gguf/clair-q5_k_m.gguf` (~2.1GB) - Better quality, more RAM
- `gguf/clair-q3_k_m.gguf` (~1.5GB) - Smaller size, lower quality

## Usage

### With llama-cpp-python
```python
from llama_cpp import Llama

llm = Llama(model_path="clair-q4_k_m.gguf", n_ctx=2048, n_threads=4)
output = llm("What is your name?", max_tokens=150)
print(output['choices'][0]['text'])
```

## Personality

Clair consistently knows:
- Its name is **Clair** (not Claude, not ChatGPT)
- Created by **Michael Mlungisi Nkomo**
- Made in **Zimbabwe**

## License

Apache 2.0 (same as base model)
"""
    
    readme_path = os.path.join(merged_model_path, "README.md")
    with open(readme_path, 'w') as f:
        f.write(model_card)
    
    api = HfApi()
    api.upload_file(
        path_or_fileobj=readme_path,
        path_in_repo="README.md",
        repo_id=repo_id,
        repo_type="model",
        commit_message="Add v2 model card",
    )
    
    print(f"✓ Model card uploaded to https://huggingface.co/{repo_id}")
else:
    print("⚠ Skipped model card upload (not logged in)")

## 📊 Summary

### Clair v2 Improvements
- ✅ 50+ training examples with varied question formats
- ✅ 20 epochs for better learning
- ✅ LoRA rank 64 for better personality capture
- ✅ Full precision training (no quantization)
- ✅ Lower learning rate (1e-4) for better convergence

### Expected Results
The model should now consistently respond with:
- **Name**: "I'm Clair" (not "I'm Claude")
- **Creator**: "Michael Mlungisi Nkomo"
- **Origin**: "Zimbabwe"